# MedImageInsight: Local Zero-Shot Image Classification

Send local images to **MedImageInsight-Premium** and compare their embeddings with candidate text descriptions.
The default candidates are **CT / H&E / X-ray / MRI**. This is an API smoke test, not disease diagnosis or treatment-response evaluation.

Preprocessing runs locally; inference uses the remote Foundry service. No Azure ML cluster is used.
For tiled H&E evaluation, see [01_aml_zero_shot_roc.ipynb](01_aml_zero_shot_roc.ipynb).

| Section | Action | Network activity |
|---|---|---|
| 1 | List images and candidate labels | None |
| 2 | Prepare and inspect PNG images | None |
| 3 | Embed candidate text | One text request |
| 4 | Embed and classify images | One request per image |
| 5 | Read saved results | None |

## Prerequisites

- Open this folder as the project root and select a Python 3.12 environment with [requirements.txt](requirements.txt) installed.
- Copy [sample.env](sample.env) to `.env` beside this notebook and configure your own deployment and tenant.
- Place authorized images in the local, Git-ignored `data/` directory.
- The client uses your existing Azure CLI sign-in through `AzureCliCredential`. Do not put credentials in source code.

**Sections 3 and 4 send requests and can incur charges.** Sections 1 and 2 are local only.

For research only. Cosine similarity is not confidence or a diagnostic probability.
PNG conversion does not remove identifiers burned into pixels. Inspect images before sending them, protect all generated artifacts,
and clear notebook outputs before publication.

In [ ]:
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import display, Image as DisplayImage

# Shared helpers are also used by the Azure ML jobs.
sys.path.insert(0, "scripts")
from mii_client import (SUPPORTED_SUFFIXES, PremiumClient, ensure_azure_cli_path, file_data_url,
                        load_labels, prepare_image, resource_endpoint, write_json)

load_dotenv(".env", override=True)

MODEL = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME") or "MedImageInsight-Premium"
ENDPOINT = resource_endpoint(os.environ["AZURE_AI_PROJECT_ENDPOINT"])
TENANT_ID = os.getenv("TENANT_ID", "")

print("Python:", sys.version.split()[0], "| kernel:", Path(sys.prefix).name)
print("Model:", MODEL)
print("Endpoint:", ENDPOINT)

## 1. List Input Images and Candidate Labels

Place PNG, JPEG, or SVS images in `data/` beside this notebook. Set `SELECTED_IMAGES` to a list of file names to select a subset.
Candidate labels and descriptions are loaded from [labels.json](labels.json).

File names are retained locally for reporting but are not included in image-inference requests.
The model receives image pixels, which may still contain identifying text.
`MAX_SIDE` limits the longest side of each prepared PNG (default: 1024 pixels).

In [ ]:
DATA_DIR = Path("data")
SELECTED_IMAGES = []          # Example: ["sample_ct.png", "sample_he.svs"]
MAX_SIDE = 1024

labels = load_labels("labels.json")
label_names = list(labels)

if SELECTED_IMAGES:
    image_paths = [DATA_DIR / name for name in SELECTED_IMAGES]
else:
    image_paths = sorted(path for path in DATA_DIR.iterdir()
                         if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES)

missing = [str(path) for path in image_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing images: {missing}")
if not image_paths:
    raise ValueError(f"No PNG, JPEG, or SVS images found in {DATA_DIR}.")

print("Images:", len(image_paths))
display(pd.DataFrame({"file_name": [path.name for path in image_paths]}))
display(pd.DataFrame(labels.items(), columns=["label", "text_prompt"]))

## 2. Prepare and Inspect PNG Images Locally

No network requests are made here. `prepare_image` converts each input to an RGB PNG.

- **PNG / JPEG:** convert to RGB and resize proportionally so that the longest side is at most `MAX_SIDE`.
- **SVS:** read an overview from the coarsest suitable tissue-pyramid level. The helper rejects files without a suitable level within its pixel limit.
  This is a slide overview, not a cellular-resolution evaluation. Use the tiled workflow in notebook 01 for that purpose.

Prepared files are written to `outputs/local-<timestamp>/prepared/`. **These are the images that will be sent to the service.**
Inspect them before continuing. Source images are not modified; identifiers visible in pixels are not automatically removed.

In [ ]:
RUN_DIR = Path("outputs") / datetime.now(timezone.utc).strftime("local-%Y%m%dT%H%M%S%fZ")
PREPARED_DIR = RUN_DIR / "prepared"
PREPARED_DIR.mkdir(parents=True, exist_ok=False)

prepared = []
for index, path in enumerate(image_paths):
    destination = PREPARED_DIR / f"{index:03d}_{path.stem}.png"
    metadata = prepare_image(path, destination, MAX_SIDE)
    metadata["file_name"] = path.name
    prepared.append(metadata)

write_json(RUN_DIR / "preprocessing.json", prepared)
print("Prepared:", RUN_DIR)
display(pd.DataFrame(prepared)[["file_name", "method", "original_size", "prepared_size"]])

for metadata in prepared:
    print(metadata["file_name"])
    display(DisplayImage(filename=metadata["prepared_image"], width=320))

## 3. Connect and Embed Candidate Text

**This section sends a request and can incur charges.**

```http
POST https://<resource>.services.ai.azure.com/providers/microsoft/v2/embed
```

Text and images are sent in separate requests. Both return 1024-dimensional embeddings.
`PremiumClient.embed` normalizes vectors to unit length, so their dot product is cosine similarity.

The following cell embeds the candidate text and checks connectivity and authorization before image requests.
A successful response has shape `(number_of_labels, 1024)`. Resolve authentication errors before proceeding.
The credential allows up to 120 seconds for the Azure CLI response. Tokens and image base64 payloads are not printed.

In [ ]:
ensure_azure_cli_path()
credential = AzureCliCredential(tenant_id=TENANT_ID or None, process_timeout=120)
client = PremiumClient(ENDPOINT, MODEL, credential)

text_vectors = client.embed("texts", [labels[name] for name in label_names])

print("text_vectors:", text_vectors.shape)   # (number_of_labels, 1024)
print("norm:", np.linalg.norm(text_vectors, axis=1))
print("API calls so far:", len(client.calls))

## 4. Embed Images and Rank Candidate Labels

1. Send each prepared image and receive its embedding.
2. Calculate the dot product with each normalized text embedding.
3. Select the candidate with the highest cosine similarity.

There is no training or threshold tuning. **A candidate is always selected, even when none describes the image correctly.**
The label set therefore defines what the classifier can predict.

The client does not automatically retry failed requests. Completed image results are saved incrementally in `results.json`.
Re-running this cell starts the image loop again; it does not resume automatically and may repeat paid requests.

In [ ]:
results = []
for metadata in prepared:
    image_vectors = client.embed("images", [file_data_url(metadata["prepared_image"])])
    similarities = np.clip(image_vectors @ text_vectors.T, -1.0, 1.0)[0]
    scores = {name: float(value) for name, value in zip(label_names, similarities)}
    winner = label_names[int(similarities.argmax())]
    results.append({"file_name": metadata["file_name"], "predicted_label": winner,
                    "cosine_similarity": scores[winner], "scores": scores})
    write_json(RUN_DIR / "results.json",
               {"model": MODEL, "labels": labels, "results": results, "api_calls": client.calls})
    print(f"{metadata['file_name']} -> {winner} (cosine={scores[winner]:.6f})", flush=True)

client.close()   # Re-run section 3 before starting another inference run.
print("Done:", RUN_DIR, "| API calls:", len(client.calls))

## 5. Read Saved Results Without Inference

The tables show the predicted candidate and the similarity to every candidate.
Cosine similarity ranges from -1 to 1. **It is not a probability or an accuracy measurement.**
Small gaps between candidates should not be interpreted as strong evidence.

To inspect a previous run, set `RESULTS_TO_VIEW` to its saved `results.json` path and run this cell.
After a kernel restart, run the import/setup cell first. Do not re-run the text or image inference sections just to inspect saved results.

In [ ]:
RESULTS_TO_VIEW = None        # Example: "outputs/local-<timestamp>/results.json"

results_file = Path(RESULTS_TO_VIEW) if RESULTS_TO_VIEW else RUN_DIR / "results.json"
saved = json.loads(results_file.read_text(encoding="utf-8"))
summary = pd.DataFrame(saved["results"])

display(summary[["file_name", "predicted_label", "cosine_similarity"]])
display(pd.DataFrame(summary["scores"].tolist(), index=summary["file_name"]))
print("Model:", saved["model"], "| results:", results_file)
print("Cosine similarity is a ranking score, not a calibrated probability.")

## Next Steps

An overview-image comparison does not inspect cellular detail across a whole slide.
The tiled workflow instead:

1. Reads tiles at a consistent physical scale.
2. Generates one embedding per selected tile.
3. Averages tile-level prompt-similarity margins to obtain an image-level score.

[01_aml_zero_shot_roc.ipynb](01_aml_zero_shot_roc.ipynb) runs this research workflow on an Azure ML CPU cluster.
The preprocessing and scoring choices still require validation for your intended dataset and task.